# NRPy 2: A 10-Minute Overview

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook gives a compact tour of NRPy 2: symbolic tensor construction with
Python and SymPy, optimized C code generation, C-function registration, and
finite-difference-aware kernel output. The running example constructs the 18
independent spatial Christoffel symbols $\Gamma^i{}_{jk}$ from a symmetric
three-metric, validates their symmetry, generates complete C artifacts,
registers one complete C function, and emits finite-difference-aware C.

Navigation: [Index](../index.ipynb) |
Previous: [Getting Started with NRPy](../0-getting_started/install_and_run.ipynb) |
Next: [Parameters](params.ipynb)


# Table of Contents

This notebook is organized as follows:

1. [Part 1](#Part-1:-Why-NRPy?): Why NRPy?
1. [Part 2](#Part-2:-Construct-spatial-Christoffels): Construct spatial
   Christoffels from the three-metric.
1. [Part 3](#Part-3:-Generate-optimized-C): Generate optimized C for the
   Christoffel expressions.
1. [Part 4](#Part-4:-Register-a-C-function): Register a C function that computes
   Christoffel symbols.
1. [Part 5](#Part-5:-Generate-finite-difference-aware-C): Generate
   finite-difference-aware C.
1. [Part 6](#Part-6:-What-next?): What next?


# Part 1: Why NRPy?
### \[Back to [top](#Table-of-Contents)\]

Numerical relativity often uses standard numerical methods, but the equations
are large, tensorial, and difficult to implement by hand without errors.

[Einstein notation](https://en.wikipedia.org/wiki/Einstein_notation) makes this
structure manageable on paper by suppressing explicit tensor sums. This notebook
uses one running example throughout: spatial Christoffel symbols
$\Gamma^i{}_{jk}$ constructed from a spatial metric $\gamma_{ij}$ and its first
derivatives.

NRPy 2 treats tensor equations written in Einstein notation as executable
symbolic specifications. In Python,

* rank-0 tensors (scalars) are *variables*
* rank-1 tensors are *lists*
* rank-2 tensors are *lists of lists*
* higher-rank tensors continue as nested lists.

NRPy tensor names encode index position. `D` means a down, or covariant, index.
`U` means an up, or contravariant, index. Thus `gammaDD` represents
$\gamma_{ij}$, `gammaUU` represents $\gamma^{ij}$, and `GammaUDD` represents
$\Gamma^i{}_{jk}$. The letter order is the tensor-index order, so
`GammaUDD[k][i][j]` stores the component $\Gamma^k{}_{ij}$.

Derivative suffixes append derivative-index groups after `_d`. A suffix `_dD`
means one derivative with a down/covariant derivative index, `_dDD` means two
derivative indices, `_dDDD` means three derivative indices, and so on. For
example, `u_dDD[i][j]` represents $\partial_i \partial_j u$, and
`gammaDD_dDD[i][j][k][l]` represents
$\partial_k \partial_l \gamma_{ij}$.

Implied tensor summations become explicit Python loops.

NRPy combines this representation with [SymPy](https://www.sympy.org/) and core
code-generation modules that convert symbolic expressions into optimized C
kernels. Its finite-difference support can also insert stencil reads directly
into generated code.

The NRPy 2 tutorial proceeds from setup through core modules, finite
differences, wave-equation examples, curvilinear coordinates, infrastructure
targets, and numerical-relativity workflows. This overview focuses on the
symbolic-to-C pipeline used throughout those notebooks.


# Part 2: Construct spatial Christoffels
### \[Back to [top](#Table-of-Contents)\]

**Problem statement**: Given a three-metric $\gamma_{ij}$, construct the 18
independent Christoffel symbols $\Gamma^i{}_{jk}$. For now, assume that
$\gamma_{ij}$ and its first derivatives are already available as symbols.

NRPy represents tensors and indexed expressions as nested Python lists. In this
notebook:

* `D` means down/covariant and `U` means up/contravariant.
* `gammaDD[i][j]` represents covariant spatial metric components
  $\gamma_{ij}$.
* `gammaUU[i][j]` represents inverse metric components $\gamma^{ij}$.
* `gammaDD_dD[i][j][k]` represents $\partial_k \gamma_{ij}$, where the trailing
  `_dD` denotes one derivative with respect to a down-index coordinate
  direction.
* Higher derivatives follow the same suffix pattern. `u_dDD[i][j]` is
  $\partial_i \partial_j u$, `betaU_dDD[i][j][k]` is
  $\partial_j \partial_k \beta^i$, and
  `gammaDD_dDDD[i][j][k][l][m]` is
  $\partial_k \partial_l \partial_m \gamma_{ij}$.
* `GammaUDD[k][i][j]` represents $\Gamma^k{}_{ij}$. The first list index is the
  contravariant `U` index, and the second and third list indices are the
  covariant `D,D` indices.

Christoffel symbols are defined by

\begin{align}
\Gamma^k{}_{ij} &= \frac{1}{2} \gamma^{kl}
    \left(\gamma_{jl,i} + \gamma_{il,j} - \gamma_{ij,l}\right).
\end{align}

The indices run over `0,1,2`. Because $\Gamma^k{}_{ij}=\Gamma^k{}_{ji}$, only
18 components are independent. First define $\gamma_{ij}$ and its inverse with
NRPy's indexed-expression helpers.


In [1]:
import sympy as sp
import nrpy.indexedexp as ixp

# gammaDD is a rank-2 indexed expression symmetric in indices 0 and 1.
gammaDD = ixp.declarerank2("gammaDD", symmetry="sym01", dimension=3)

# gammaUU is the inverse of gammaDD; gammaDET is its determinant.
gammaUU, gammaDET = ixp.symm_matrix_inverter3x3(gammaDD)

The generated objects encode the requested symmetry. The inverse metric is also
symmetric, and $\gamma_{ij}\gamma^{ji}=3$ for a three-dimensional metric. The
next cell prints representative components and verifies these identities.

**Exercise**: Replace the final trace check below with explicit loops over all
dimensions, then simplify the result.


In [2]:
print("Check that gammaDD[0][1] = gammaDD[1][0]:")
print("gammaDD[0][1] = ", gammaDD[0][1])
print("gammaDD[1][0] = ", gammaDD[1][0])
assert gammaDD[0][1] == gammaDD[1][0]

print("\nOutput gammaUU[1][0] = ", gammaUU[1][0])
print("\nCheck that gammaUU[1][0] - gammaUU[0][1] = 0:")
difference = sp.simplify(gammaUU[1][0] - gammaUU[0][1])
print("gammaUU[1][0] - gammaUU[0][1] = ", difference)
assert difference == 0

trace = sp.simplify(sum(gammaDD[i][j] * gammaUU[j][i] for i in range(3) for j in range(3)))
print("\nTrace gammaDD[i][j] * gammaUU[j][i] = ", trace)
assert trace == 3

Check that gammaDD[0][1] = gammaDD[1][0]:
gammaDD[0][1] =  gammaDD01
gammaDD[1][0] =  gammaDD01

Output gammaUU[1][0] =  (-gammaDD01*gammaDD22 + gammaDD02*gammaDD12)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*gammaDD12**2 - gammaDD01**2*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - gammaDD02**2*gammaDD11)

Check that gammaUU[1][0] - gammaUU[0][1] = 0:
gammaUU[1][0] - gammaUU[0][1] =  0

Trace gammaDD[i][j] * gammaUU[j][i] =  3


The next cell defines the first derivatives `gammaDD_dD` and builds
`GammaUDD`. The nested loops implement the implied sum over $l$ in

$$
\Gamma^k{}_{ij} = \frac{1}{2} \gamma^{kl}
\left(\gamma_{jl,i} + \gamma_{il,j} - \gamma_{ij,l}\right).
$$

The output prints one complete component so the mapping from the indexed
equation to the symbolic expression is visible before the symmetry check.


In [3]:
# First define symbolic expressions for metric derivatives.
gammaDD_dD = ixp.declarerank3("gammaDD_dD", symmetry="sym01", dimension=3)

# Initialize GammaUDD (spatial Christoffel symbols) to zero.
GammaUDD = ixp.zerorank3(dimension=3)
for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                GammaUDD[k][i][j] += sp.Rational(1, 2) * gammaUU[k][l] * (
                    gammaDD_dD[j][l][i]
                    + gammaDD_dD[i][l][j]
                    - gammaDD_dD[i][j][l]
                )

print("Example component GammaUDD[0][0][0] = Gamma^0{}_{00}:")
print(GammaUDD[0][0][0])


Example component GammaUDD[0][0][0] = Gamma^0{}_{00}:
gammaDD_dD000*(gammaDD11*gammaDD22 - gammaDD12**2)/(2*(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*gammaDD12**2 - gammaDD01**2*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - gammaDD02**2*gammaDD11)) + (-gammaDD_dD001 + 2*gammaDD_dD010)*(-gammaDD01*gammaDD22 + gammaDD02*gammaDD12)/(2*(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*gammaDD12**2 - gammaDD01**2*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - gammaDD02**2*gammaDD11)) + (-gammaDD_dD002 + 2*gammaDD_dD020)*(gammaDD01*gammaDD12 - gammaDD02*gammaDD11)/(2*(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*gammaDD12**2 - gammaDD01**2*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - gammaDD02**2*gammaDD11))


Now confirm the lower-index symmetry $\Gamma^k{}_{ij}=\Gamma^k{}_{ji}$. This
checks every component rather than only the displayed example.


In [4]:
assert all(
    sp.simplify(GammaUDD[k][i][j] - GammaUDD[k][j][i]) == 0
    for i in range(3)
    for j in range(3)
    for k in range(3)
)
print("All GammaUDD[k][i][j] - GammaUDD[k][j][i] symmetry checks vanish.")

All GammaUDD[k][i][j] - GammaUDD[k][j][i] symmetry checks vanish.


# Part 3: Generate optimized C
### \[Back to [top](#Table-of-Contents)\]

At the core of NRPy is the ability to convert SymPy expressions to optimized C
code.

**Problem statement**: Output all 18 unique Christoffel symbols with three
optimization modes supported by NRPy's `c_codegen()`.

First store all 18 unique Christoffel symbols, their desired C variable names,
and their flat component order in Python lists.


In [5]:
symbols_list = []
varname_list = []
component_index_map = []
for i in range(3):
    for j in range(3):
        for k in range(j, 3):
            symbols_list.append(GammaUDD[i][j][k])
            varname_list.append(f"GammaUDD{i}{j}{k}")
            component_index_map.append((i, j, k))

assert len(symbols_list) == 18
assert len(varname_list) == 18
assert len(component_index_map) == 18

print(f"Prepared {len(symbols_list)} independent Christoffel expressions.")
print("Flat component order used below:")
for component, (i, j, k) in enumerate(component_index_map):
    print(f"  GammaUDD[{component:2d}] -> Gamma^{i}{{}}_{{{j}{k}}}")


Prepared 18 independent Christoffel expressions.
Flat component order used below:
  GammaUDD[ 0] -> Gamma^0{}_{00}
  GammaUDD[ 1] -> Gamma^0{}_{01}
  GammaUDD[ 2] -> Gamma^0{}_{02}
  GammaUDD[ 3] -> Gamma^0{}_{11}
  GammaUDD[ 4] -> Gamma^0{}_{12}
  GammaUDD[ 5] -> Gamma^0{}_{22}
  GammaUDD[ 6] -> Gamma^1{}_{00}
  GammaUDD[ 7] -> Gamma^1{}_{01}
  GammaUDD[ 8] -> Gamma^1{}_{02}
  GammaUDD[ 9] -> Gamma^1{}_{11}
  GammaUDD[10] -> Gamma^1{}_{12}
  GammaUDD[11] -> Gamma^1{}_{22}
  GammaUDD[12] -> Gamma^2{}_{00}
  GammaUDD[13] -> Gamma^2{}_{01}
  GammaUDD[14] -> Gamma^2{}_{02}
  GammaUDD[15] -> Gamma^2{}_{11}
  GammaUDD[16] -> Gamma^2{}_{12}
  GammaUDD[17] -> Gamma^2{}_{22}


Next pass these lists to NRPy's C code-generation function `c_codegen()`.

**Optimization Level 0**: compute each Christoffel symbol independently. Inspect
the repeated metric-determinant denominator and the direct assignment to every
`GammaUDDijk` output variable.


In [6]:
import nrpy.c_codegen as ccg

code_no_cse = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=False,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
for varname in varname_list:
    assert f"{varname} =" in code_no_cse

print("Complete generated C without CSE:")
print(code_no_cse)


Complete generated C without CSE:
GammaUDD000 = (1.0/2.0)*gammaDD_dD000*(gammaDD11*gammaDD22 - ((gammaDD12)*(gammaDD12)))/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11) + (1.0/2.0)*(-gammaDD_dD001 + 2*gammaDD_dD010)*(-gammaDD01*gammaDD22 + gammaDD02*gammaDD12)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11) + (1.0/2.0)*(-gammaDD_dD002 + 2*gammaDD_dD020)*(gammaDD01*gammaDD12 - gammaDD02*gammaDD11)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11);
GammaUDD001 = (1.0/2.0)*gammaDD_dD001*(gammaDD11*gammaDD22 - ((gammaDD12)*(gammaDD12)))/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12

The unoptimized code recomputes many factors. Common subexpression elimination
introduces temporaries for repeated symbolic subexpressions.

**Optimization Level 1**: use
[common subexpression elimination](https://en.wikipedia.org/wiki/Common_subexpression_elimination).
Inspect the `tmp` variables first, then the final `GammaUDDijk`
assignments that reuse them.


In [7]:
code_cse = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
for varname in varname_list:
    assert f"{varname} =" in code_cse
assert "const REAL tmp" in code_cse

print("Complete generated C with CSE:")
print(code_cse)


Complete generated C with CSE:
const REAL tmp5 = -gammaDD_dD001 + 2*gammaDD_dD010;
const REAL tmp7 = -gammaDD_dD002 + 2*gammaDD_dD020;
const REAL tmp9 = -gammaDD_dD012 + gammaDD_dD021 + gammaDD_dD120;
const REAL tmp10 = gammaDD_dD012 - gammaDD_dD021 + gammaDD_dD120;
const REAL tmp11 = -gammaDD_dD112 + 2*gammaDD_dD121;
const REAL tmp12 = 2*gammaDD_dD011 - gammaDD_dD110;
const REAL tmp13 = gammaDD_dD012 + gammaDD_dD021 - gammaDD_dD120;
const REAL tmp14 = 2*gammaDD_dD122 - gammaDD_dD221;
const REAL tmp15 = 2*gammaDD_dD022 - gammaDD_dD220;
const REAL tmp3 = (1.0/2.0)/(gammaDD00*gammaDD11*gammaDD22 - gammaDD00*((gammaDD12)*(gammaDD12)) - ((gammaDD01)*(gammaDD01))*gammaDD22 + 2*gammaDD01*gammaDD02*gammaDD12 - ((gammaDD02)*(gammaDD02))*gammaDD11);
const REAL tmp4 = tmp3*(gammaDD11*gammaDD22 - ((gammaDD12)*(gammaDD12)));
const REAL tmp6 = tmp3*(-gammaDD01*gammaDD22 + gammaDD02*gammaDD12);
const REAL tmp8 = tmp3*(gammaDD01*gammaDD12 - gammaDD02*gammaDD11);
const REAL tmp16 = tmp3*(-gammaDD00*ga

**Optimization Level 2**: use CSE and
[single instruction, multiple data](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data)
helper macros. This form is intended for kernels evaluated over numerical grid
data. Inspect the SIMD constants, SIMD temporaries, and final SIMD assignment
calls.


In [8]:
code_simd = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=True,
    enable_simd=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
for varname in varname_list:
    assert f"{varname} =" in code_simd
assert "SIMD" in code_simd

print("Complete generated C with CSE plus SIMD:")
print(code_simd)


Complete generated C with CSE plus SIMD:
static const double dbl_Integer_1 = 1.0;
MAYBE_UNUSED const REAL_SIMD_ARRAY _Integer_1 = ConstSIMD(dbl_Integer_1);

static const double dbl_Integer_2 = 2.0;
 const REAL_SIMD_ARRAY _Integer_2 = ConstSIMD(dbl_Integer_2);

static const double dbl_NegativeOne_ = -1.0;
MAYBE_UNUSED const REAL_SIMD_ARRAY _NegativeOne_ = ConstSIMD(dbl_NegativeOne_);

static const double dbl_Rational_1_2 = 1.0/2.0;
 const REAL_SIMD_ARRAY _Rational_1_2 = ConstSIMD(dbl_Rational_1_2);

const REAL_SIMD_ARRAY tmp1 = MulSIMD(_NegativeOne_, MulSIMD(gammaDD12, gammaDD12));
const REAL_SIMD_ARRAY tmp3 = MulSIMD(_NegativeOne_, MulSIMD(gammaDD01, gammaDD01));
const REAL_SIMD_ARRAY tmp4 = MulSIMD(_NegativeOne_, MulSIMD(gammaDD02, gammaDD02));
const REAL_SIMD_ARRAY tmp7 = FusedMulSubSIMD(_Integer_2, gammaDD_dD010, gammaDD_dD001);
const REAL_SIMD_ARRAY tmp9 = FusedMulSubSIMD(_Integer_2, gammaDD_dD020, gammaDD_dD002);
const REAL_SIMD_ARRAY tmp11 = AddSIMD(gammaDD_dD120, SubSIMD(gammaDD

# Part 4: Register a C function
### \[Back to [top](#Table-of-Contents)\]

Generated assignments become more useful when they are placed inside a complete C
function. A C function has a fixed outer structure:

```c
/*
 * Description of what the function computes.
 */
return_type function_name(parameter_list) {
    local_declarations;
    generated_assignments;
}
```

NRPy's `register_CFunction()` mirrors this structure. The function description
maps to `desc`, the return type maps to `cfunc_type`, the C name maps to `name`,
the input/output list maps to `params`, and the generated assignment block maps
to `body`. The `includes` list records headers needed when the generated project
writes this function to source files.

The registered function below takes six independent metric components, eighteen
first-derivative inputs for the symmetric `gammaDD_dD`, and one output pointer
`REAL *restrict GammaUDD`. The flat output order is the `component_index_map`
printed above: `GammaUDD[n]` stores the component $\Gamma^i{}_{jk}$ listed on
line `n` of that map.

Across BHaH registrations, this same approach appears repeatedly: define the
C-function pieces as Python variables, generate or assemble the body, then pass
those variables directly to `register_CFunction()`. The registration helper does
not print or inspect the stored `CFunction` object. Tutorial evidence belongs in
the following inspection cell.


In [9]:
import nrpy.c_function as cfc


def register_CFunction_compute_Christoffel_symbols() -> None:
    """Register a C function for the independent Christoffel symbols."""
    christoffel_output_varnames = [
        f"GammaUDD[{component}]"
        for component in range(len(component_index_map))
    ]
    body = ccg.c_codegen(
        symbols_list,
        christoffel_output_varnames,
        enable_cse=True,
        verbose=False,
        include_braces=False,
    )

    includes = ["BHaH_defines.h"]
    desc = "Compute the 18 independent spatial Christoffel symbols Gamma^i{}_{jk}."
    cfunc_type = "void"
    name = "compute_Christoffel_symbols"
    metric_params = [
        "const REAL gammaDD00",
        "const REAL gammaDD01",
        "const REAL gammaDD02",
        "const REAL gammaDD11",
        "const REAL gammaDD12",
        "const REAL gammaDD22",
    ]
    deriv_params = [
        f"const REAL gammaDD_dD{i}{j}{k}"
        for i in range(3)
        for j in range(i, 3)
        for k in range(3)
    ]
    output_params = ["REAL *restrict GammaUDD"]
    params = ", ".join(metric_params + deriv_params + output_params)
    include_CodeParameters_h = False

    cfc.register_CFunction(
        includes=includes,
        desc=desc,
        cfunc_type=cfunc_type,
        name=name,
        params=params,
        include_CodeParameters_h=include_CodeParameters_h,
        body=body,
    )


In [10]:
christoffel_cfunc_name = "compute_Christoffel_symbols"

# Keep this tutorial cell idempotent for manual reruns. This is outside the
# registration helper so the helper still matches the BHaH registration pattern.
cfc.CFunction_dict.pop(christoffel_cfunc_name, None)
register_CFunction_compute_Christoffel_symbols()

registered = cfc.CFunction_dict[christoffel_cfunc_name]
expected_metric_params = [
    "const REAL gammaDD00",
    "const REAL gammaDD01",
    "const REAL gammaDD02",
    "const REAL gammaDD11",
    "const REAL gammaDD12",
    "const REAL gammaDD22",
]
expected_deriv_params = [
    f"const REAL gammaDD_dD{i}{j}{k}"
    for i in range(3)
    for j in range(i, 3)
    for k in range(3)
]
expected_output_params = ["REAL *restrict GammaUDD"]
expected_params = ", ".join(
    expected_metric_params + expected_deriv_params + expected_output_params
)

assert registered.name == christoffel_cfunc_name
assert registered.includes == ["BHaH_defines.h"]
assert registered.params == expected_params
assert registered.include_CodeParameters_h is False
for component in range(len(component_index_map)):
    assert f"GammaUDD[{component}] =" in registered.body

print("Registered C function metadata:")
print("name:", registered.name)
print("includes:", registered.includes)
print("include_CodeParameters_h:", registered.include_CodeParameters_h)
print("parameter count:", len(registered.params.split(", ")))

print("\nFlat output order:")
for component, (i, j, k) in enumerate(component_index_map):
    print(f"  GammaUDD[{component:2d}] -> Gamma^{i}{{}}_{{{j}{k}}}")

print("\nComplete registered C function:")
print(registered.full_function)


Registered C function metadata:
name: compute_Christoffel_symbols
includes: ['BHaH_defines.h']
include_CodeParameters_h: False
parameter count: 25

Flat output order:
  GammaUDD[ 0] -> Gamma^0{}_{00}
  GammaUDD[ 1] -> Gamma^0{}_{01}
  GammaUDD[ 2] -> Gamma^0{}_{02}
  GammaUDD[ 3] -> Gamma^0{}_{11}
  GammaUDD[ 4] -> Gamma^0{}_{12}
  GammaUDD[ 5] -> Gamma^0{}_{22}
  GammaUDD[ 6] -> Gamma^1{}_{00}
  GammaUDD[ 7] -> Gamma^1{}_{01}
  GammaUDD[ 8] -> Gamma^1{}_{02}
  GammaUDD[ 9] -> Gamma^1{}_{11}
  GammaUDD[10] -> Gamma^1{}_{12}
  GammaUDD[11] -> Gamma^1{}_{22}
  GammaUDD[12] -> Gamma^2{}_{00}
  GammaUDD[13] -> Gamma^2{}_{01}
  GammaUDD[14] -> Gamma^2{}_{02}
  GammaUDD[15] -> Gamma^2{}_{11}
  GammaUDD[16] -> Gamma^2{}_{12}
  GammaUDD[17] -> Gamma^2{}_{22}

Complete registered C function:
#include "BHaH_defines.h"

/**
 * Compute the 18 independent spatial Christoffel symbols Gamma^i{}_{jk}.
 */
void compute_Christoffel_symbols(const REAL gammaDD00, const REAL gammaDD01, const REAL gammaDD02

# Part 5: Generate finite-difference-aware C
### \[Back to [top](#Table-of-Contents)\]

The previous C code assumed that $\gamma_{ij,k}$ values had already been
computed. NRPy's finite-difference-aware code generation can instead insert
stencil expressions for derivative symbols.

Gridfunctions are fields stored on the numerical grid. Here the input metric
components are registered as evolution gridfunctions, and the independent
Christoffel components are registered as auxiliary output gridfunctions.

The parameter `fd_order=6` selects sixth-order finite-difference stencils.
Derivative names such as `gammaDD_dD000` use the same suffix convention defined
above: `D` denotes the down/covariant derivative direction. Higher-derivative
names extend the pattern. For example, `u_dDD01` has two derivative letters
after `_d`; the trailing digits say that those derivative indices point in
coordinate directions `0` and `1`.

`Infrastructure="BHaH"` selects the BHaH memory-access convention. Inspect the
complete output for the `IDX4` grid reads, the derivative temporary
`gammaDD_dD000`, and the 18 assignments to auxiliary Christoffel gridfunctions.


In [11]:
import nrpy.grid as gri
import nrpy.params as par

gri.glb_gridfcs_dict.clear()
par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fd_order", 6)

gri.register_gridfunctions_for_single_rank2(
    "gammaDD", group="EVOL", dimension=3, symmetry="sym01"
)

christoffel_gf_names = list(varname_list)
gri.register_gridfunctions(
    christoffel_gf_names,
    group="AUX",
    dimension=3,
    rank=3,
    is_basename=False,
)

output_varnames = [
    gri.BHaHGridFunction.access_gf(gf_name=name, gf_array_name="aux_gfs")
    for name in christoffel_gf_names
]
fd_code = ccg.c_codegen(
    symbols_list,
    output_varnames,
    enable_cse=True,
    enable_fd_codegen=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)

assert "NRPy-Generated GF Access/FD Code" in fd_code
assert "IDX4" in fd_code
assert "gammaDD_dD000" in fd_code
for name in christoffel_gf_names:
    assert f"{name.upper()}GF" in fd_code

print("Complete finite-difference-aware C code:")
print(fd_code)


Complete finite-difference-aware C code:
/*
 * NRPy-Generated GF Access/FD Code, Step 1 of 2:
 * Read gridfunction(s) from main memory and compute FD stencils as needed.
 */
const REAL gammaDD00_i2m3 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2-3)];
const REAL gammaDD00_i2m2 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2-2)];
const REAL gammaDD00_i2m1 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2-1)];
const REAL gammaDD00_i1m3 = in_gfs[IDX4(GAMMADD00GF, i0, i1-3, i2)];
const REAL gammaDD00_i1m2 = in_gfs[IDX4(GAMMADD00GF, i0, i1-2, i2)];
const REAL gammaDD00_i1m1 = in_gfs[IDX4(GAMMADD00GF, i0, i1-1, i2)];
const REAL gammaDD00_i0m3 = in_gfs[IDX4(GAMMADD00GF, i0-3, i1, i2)];
const REAL gammaDD00_i0m2 = in_gfs[IDX4(GAMMADD00GF, i0-2, i1, i2)];
const REAL gammaDD00_i0m1 = in_gfs[IDX4(GAMMADD00GF, i0-1, i1, i2)];
const REAL gammaDD00 = in_gfs[IDX4(GAMMADD00GF, i0, i1, i2)];
const REAL gammaDD00_i0p1 = in_gfs[IDX4(GAMMADD00GF, i0+1, i1, i2)];
const REAL gammaDD00_i0p2 = in_gfs[IDX4(GAMMADD00GF, i0+2, i1, i2)];
const

# Part 6: What next?
### \[Back to [top](#Table-of-Contents)\]

This overview constructed $\Gamma^i{}_{jk}$ symbolically, validated its lower-index
symmetry, printed complete generated C in several optimization modes, registered
and inspected a complete C function, and generated finite-difference-aware C for
the same running example.

Continue with the notebooks most relevant to your next task:

- [Repository index](../index.ipynb)
- [Parameters](params.ipynb)
- [Indexed expressions](indexedexp.ipynb)
- [Grid fields](grid.ipynb)
- [C code generation](c_codegen.ipynb)
- [C function registration](c_function.ipynb)
- [Finite differences](finite_difference.ipynb)
- [Wave equation with NumPy](../2-numerical_methods/wave_equation_with_numpy.ipynb)
- [Wave equation with C code generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)
